In [2]:
# Pt 3. Molecule Descriptor Calculation Using PaDEL
# PaDEL: Remove salt and small organic acid, standardized the chemical stucture, compute fingerprint 
# with PubChem type and upload it to descriptor_output.csv

import pandas as pd
import urllib.request
import zipfile
import os
from pathlib import Path

folder = Path(
    r"D:\ANDINA\Course\Bioinformatics\Python\EGFR Drug Discovery"
)

In [3]:
%pip install padelpy

  Using cached padelpy-0.1.17-py3-none-any.whl.metadata (6.5 kB)
Using cached padelpy-0.1.17-py3-none-any.whl (20.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Check java version
!java -version

openjdk version "25.0.4.1" 2026-08-18 LTS
OpenJDK Runtime Environment Temurin-25.0.4.1+1 (build 25.0.4.1+1-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.4.1+1 (build 25.0.4.1+1-LTS, mixed mode, sharing)


In [5]:
# Check the PaDELpy import
from padelpy import padeldescriptor
print("PaDELpy imported successfully!")

PaDELpy imported successfully!


In [7]:
# Open the CSV file and read it into a DataFrame
df = pd.read_csv('bioactivity_data_final.csv')
df

,molecule_chembl_id,canonical_smiles,class,MolWt,MolLogP,NumHDonors,NumHAcceptors,pIC50
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,active,383.814,4.45034,3.0,4.0,7.387216
1,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,active,482.903,3.61432,3.0,6.0,6.769551
2,CHEMBL137635,CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12,intermediate,369.432,4.77200,1.0,6.0,5.031517
3,CHEMBL306988,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)O)cc1,inactive,283.287,2.31056,2.0,4.0,3.301030
4,CHEMBL66879,O=C(O)/C=C/c1ccc(O)cc1,inactive,164.160,1.49000,2.0,2.0,2.522879
...,...,...,...,...,...,...,...,...
786,CHEMBL443296,CCOc1cccc(-c2ccc3c(c2)NC(=O)/C3=C\c2[nH]cc(C)c...,inactive,416.477,4.89872,3.0,3.0,4.000000
787,CHEMBL149243,COc1ccc2c(c1)NC(=O)/C2=C\c1[nH]cc(CCC(=O)O)c1C,inactive,326.352,2.84162,3.0,3.0,4.000000
788,CHEMBL149073,Cc1[nH]c(/C=C2\C(=O)Nc3ccc(C(=O)O)cc32)c(C)c1C...,inactive,354.362,2.83964,4.0,3.0,4.000000
789,CHEMBL148984,COc1ccc(-c2ccc3c(c2)NC(=O)/C3=C\c2[nH]cc(CCC(=...,inactive,402.450,4.50862,3.0,3.0,4.000000


In [8]:
# Prepare the input file for PaDEL
selection = ['canonical_smiles','molecule_chembl_id', 'pIC50']
df3_padel = df[selection]
df3_padel.to_csv('molecule.smi', sep='\t', index=False, header=False)
df3_padel.head(5)

,canonical_smiles,molecule_chembl_id,pIC50
0,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,CHEMBL68920,7.387216
1,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,CHEMBL69960,6.769551
2,CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12,CHEMBL137635,5.031517
3,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)O)cc1,CHEMBL306988,3.301030
4,O=C(O)/C=C/c1ccc(O)cc1,CHEMBL66879,2.522879


In [9]:
# Prepare the input file for PaDEL
selection = ['canonical_smiles','molecule_chembl_id']
df3_selection = df[selection]
df3_selection.to_csv('molecule.smi', sep='\t', index=False, header=False)
df3_selection.head(5)

,canonical_smiles,molecule_chembl_id
0,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,CHEMBL68920
1,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,CHEMBL69960
2,CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12,CHEMBL137635
3,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)O)cc1,CHEMBL306988
4,O=C(O)/C=C/c1ccc(O)cc1,CHEMBL66879


In [10]:
# Download the fingerprint configuration files
xml_zip = folder / "fingerprints_xml.zip"
xml_folder = folder / "fingerprints_xml"

url = (
    "https://raw.githubusercontent.com/"
    "dataprofessor/padel/main/fingerprints_xml.zip"
)

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)

with urllib.request.urlopen(request, timeout=60) as response:
    xml_zip.write_bytes(response.read())

print("Downloaded size:", xml_zip.stat().st_size, "bytes")
print("Valid ZIP:", zipfile.is_zipfile(xml_zip))

Downloaded size: 10871 bytes
Valid ZIP: True


In [11]:
# Extract the XML files
xml_folder.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(xml_zip, "r") as z:
    z.extractall(xml_folder)

xml_path = xml_folder / "PubchemFingerprinter.xml"

print("PubChem XML exists:", xml_path.exists())
print("XML path:", xml_path)

PubChem XML exists: True
XML path: D:\ANDINA\Course\Bioinformatics\Python\EGFR Drug Discovery\fingerprints_xml\PubchemFingerprinter.xml


In [12]:
# Creating a .smi file containing each molecule's SMILE and its identifier

selection = [
    "canonical_smiles",
    "molecule_chembl_id"
]

df3_selection = df3_padel[selection].copy()

molecule_path = folder / "molecule.smi"

df3_selection.to_csv(
    molecule_path,
    sep="\t",
    index=False,
    header=False
)

print("Saved:", molecule_path)
print("Number of molecules:", len(df3_selection))

Saved: D:\ANDINA\Course\Bioinformatics\Python\EGFR Drug Discovery\molecule.smi
Number of molecules: 791


In [13]:
# Inspect the result file
with molecule_path.open("r", encoding="utf-8") as f:
    for _, line in zip(range(5), f):
        print(line, end="")

print("\nTotal lines:", sum(1 for _ in molecule_path.open(
    "r", encoding="utf-8"
)))

Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)c32)[nH]1	CHEMBL68920
Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(Nc3ccc(F)c(Cl)c3)c21	CHEMBL69960
CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12	CHEMBL137635
CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)O)cc1	CHEMBL306988
O=C(O)/C=C/c1ccc(O)cc1	CHEMBL66879

Total lines: 791


In [14]:
# Calculate PubChem fingerprints using PaDELpy

from padelpy import padeldescriptor

descriptor_path = folder / "descriptors_output.csv"

padeldescriptor(
    mol_dir=str(molecule_path),
    d_file=str(descriptor_path),
    descriptortypes=str(xml_path),
    fingerprints=True,
    removesalt=True,
    standardizenitro=True,
    detectaromaticity=True,
    standardizetautomers=True,
    retainorder=True,
    threads=2,
    log=True,
    headless=True
)

print("Descriptor calculation finished.")
print("Output exists:", descriptor_path.exists())

Descriptor calculation finished.
Output exists: True


In [16]:
# Check the result
if not descriptor_path.exists():
    raise FileNotFoundError(
        "PaDEL did not create descriptors_output.csv"
    )

df3_descriptors = pd.read_csv(descriptor_path)

print("Descriptor table shape:", df3_descriptors.shape)
print("First columns:", df3_descriptors.columns[:10].tolist())

df3_descriptors

Descriptor table shape: (791, 882)
First columns: ['Name', 'PubchemFP0', 'PubchemFP1', 'PubchemFP2', 'PubchemFP3', 'PubchemFP4', 'PubchemFP5', 'PubchemFP6', 'PubchemFP7', 'PubchemFP8']


,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL68920,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL69960,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL137635,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL306988,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL66879,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
786,CHEMBL443296,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
787,CHEMBL149243,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
788,CHEMBL149073,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
789,CHEMBL148984,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
# Prepare the descriptor matrix consisting of X and Y
# X: contains molecular fingerprint features calculated by PaDEL
# Y: contains the pIC50 value that the model have to predict

if "Name" not in df3_descriptors.columns:
    raise ValueError(
        "The descriptor output has no 'Name' column. "
        "Inspect df3_descriptors.head() before continuing."
    )

df3_descriptors["Name"] = (
    df3_descriptors["Name"].astype(str).str.strip()
)

df3_X_with_id = df3_descriptors.rename(
    columns={"Name": "molecule_chembl_id"}
)

df3_X_with_id.head()

,molecule_chembl_id,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL68920,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL69960,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL137635,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL306988,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL66879,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
df3_Y = df3_padel[
    ["molecule_chembl_id", "pIC50"]
].copy()

df3_Y.head()

,molecule_chembl_id,pIC50
0,CHEMBL68920,7.387216
1,CHEMBL69960,6.769551
2,CHEMBL137635,5.031517
3,CHEMBL306988,3.301030
4,CHEMBL66879,2.522879


In [19]:
# Match X and Y by molecule ID
# Check whether each molecule ID appears only once
if df3_X_with_id["molecule_chembl_id"].duplicated().any():
    raise ValueError(
        "Duplicate molecule IDs found in the PaDEL output. "
        "Inspect the duplicate IDs before merging."
    )

if df3_Y["molecule_chembl_id"].duplicated().any():
    raise ValueError(
        "Duplicate molecule IDs found in the target data. "
        "Decide how to handle repeated activity measurements first."
    )

matched = pd.merge(
    df3_X_with_id,
    df3_Y,
    on="molecule_chembl_id",
    how="inner",
    validate="one_to_one"
)

print("Matched molecules:", len(matched))
print("Original prepared molecules:", len(df3_padel))
print("Unmatched molecules:", len(df3_padel) - len(matched))

Matched molecules: 791
Original prepared molecules: 791
Unmatched molecules: 0


In [20]:
# Create X, Y, and the combined dataset
# X: molecular fingerprint features only
df3_X = matched.drop(
    columns=["molecule_chembl_id", "pIC50"]
)

# Y: biological activity target
df3_Y = matched[["pIC50"]]

# Combine X and Y into one machine-learning dataset
dataset3 = pd.concat(
    [df3_X.reset_index(drop=True),
     df3_Y.reset_index(drop=True)],
    axis=1
)

print("X matrix shape:", df3_X.shape)
print("Y matrix shape:", df3_Y.shape)
print("Combined dataset shape:", dataset3.shape)

dataset3.head()

X matrix shape: (791, 881)
Y matrix shape: (791, 1)
Combined dataset shape: (791, 882)


,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,...,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880,pIC50
0,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.387216
1,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,6.769551
2,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,5.031517
3,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,3.301030
4,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,2.522879


In [ ]:
# Save the combined dataset
dataset3.to_csv(folder / "bioactivity_XY_dataset.csv", index=False)

print("Saved dataset file to:", folder)

Saved all three files to: D:\ANDINA\Course\Bioinformatics\Python\EGFR Drug Discovery
